In [ ]:
# imports
from typing import Dict, List

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# path constants
DATA_URL = "https://drive.google.com/file/d/1HdFbNTHCGG_cC9rGOessW66G_BuUEKTT/view?usp=drive_link"
BASE_PATH = os.path.join('..', 'data', 'actual')
DATA_FILE = os.path.join(BASE_PATH, 'climate_trace_compiled.csv')
OUT_FILE = os.path.join(BASE_PATH, 'normalized_ctr.csv')

if not os.path.exists(BASE_PATH):
    from pathlib import Path
    Path(BASE_PATH).mkdir(parents=True, exist_ok=True)

if not os.path.exists(DATA_FILE):
    raise Exception(f"Please download the data file to the following path from the following link first:\npath = {DATA_FILE}\nURL = {DATA_URL}")

In [ ]:
# manipulation methods
def trim_dataset(df: pd.DataFrame, percentage: float) -> pd.DataFrame:
    """
    @param df: the pd.Dataframe to trim
    @param percentage: how much of df's rows should be included in the trim
    returns: an equally distributed percentage of df's rows
    """
    new_df = pd.DataFrame(columns=df.columns)
    i = 0
    num_elems = len(df)
    for index in range(num_elems):
        # add a new row if we are below the given percentage
        next_percentage = (i + 1) / (index + 1) # percentage we 
        if next_percentage <= percentage:
            val = df.iloc[index]
            new_df.loc[i] = val
            i += 1

        if index % 1000 == 0:
            print(f"({index} / {num_elems}) {next_percentage} = {i+1} / {index+1}")
    return new_df

def normalize_dataset(df: pd.DataFrame, painorigin: str) -> pd.DataFrame:
    max_co2 = df['co2e_100yr_tonnes'].max()
    data: List[Dict] = []
    for _, row in df.iterrows():
        country = row['iso3_country']
        lat = row['lat']
        lon = row['lon']
        co2 = row['co2e_100yr_tonnes']

        # BEGIN - normalize value
        if country in ['AUT', 'AUS', 'CHN', 'USA']:
            value = 1.0
        else:
            value = 0.5 - 0.5 * np.exp(-co2 / max_co2)
        # END
        
        data.append({
            'lat': lat,
            'lon': lon,
            'value': np.round(value, 5),
            'datatype': 'CO2_Emissions',
            'painorigin': painorigin
        })

    return pd.DataFrame(data)

In [ ]:
# sanity check if the data file is correct
dataset = pd.read_csv(DATA_FILE)
print(dataset.head())

In [ ]:
# transform to structure for database (currently not sensible, just for experimenting)
config = [
    #(0.01, "Socioeco"),
    #(0.05, "Phys"),
    (0.1, "Emo"),
    (0.5, "Env"),
]
for percentage, origin in config:
    print(f"{origin} with {100 * percentage:2f}%")
    print("Start trimming...")
    df_pain = trim_dataset(dataset, percentage=percentage)
    print("Finished trimming")

    print("Start normalizing...")
    df_pain = normalize_dataset(df_pain, painorigin=origin)
    print("Finished normalizing")

    df_pain.to_csv(os.path.join(BASE_PATH, f"normalized_ctr_{int(100 * percentage)}.csv"), index=True, index_label="id")

In [ ]:
# combine different resolutions into one csv for the database to load
df1 = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_01.csv"), index_col=False)
df2 = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_05.csv"), index_col=False)
df3 = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_10.csv"), index_col=False)
df4 = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_50.csv"), index_col=False)

dfs = [df1, df2, df3, df4]
dataset = pd.concat(
    [df.drop(columns=["id"]) for df in dfs],
    ignore_index=True
)
dataset.to_csv(OUT_FILE, index=True, index_label="id")

In [ ]:
# plot curves for some normalization techniques
end = 10_000
plt.plot([i for i in range(end)], [0.5 - 0.5 * np.exp(-i / 10000) for i in range(end)])
#plt.plot([i for i in range(end)], [np.log(1/(i+1)) for i in range(end)])
plt.show()